### 절대 모멘텀

- 전년도(3개월, 6개월, 12개월)의 수정 주가와 전월의 수정 주가를 이용하여 구매의 타이밍을 잡는 투자 전략
- 구매 신호 → (전월의 수정 주가 / 전년도의 수정 주가) - 1
    - 값이 0보다 크고 무한대가 아닌 경우

1. 파생변수 STD-YM 생성 → index에서 년-월을 추출하여 대입
2. STD-YM 별 마지막 날의 데이터들을 모아서 month_last_df 데이터프레임을 생성
3. 전월의 수정주가 파생변수 생성 → 전월의 수정 주가 대입
4. 전년도의 수정주가 파생변수 생성 → 전년도의 수정 주가 대입
5. 구매 신호 생성
6. 원본의 데이터에서 구호 신호에 따른 거래 내역 생성
7. 수익률 계산

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
df = pd.read_csv('../csv/AMZN.csv', index_col='Date')
df.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200


In [3]:
# index 시계열로 변환
df.index = pd.to_datetime(df.index)

In [4]:
# index 데이터에서 년-월을 추출하여 STD-YM에 대입
df['STD-YM'] = df.index.strftime('%Y-%m')

In [5]:
df.head()

,Open,High,Low,Close,Adj Close,Volume,STD-YM
Date,,,,,,,
1997-05-15,2.437500,2.500000,1.927083,1.958333,1.958333,72156000,1997-05
1997-05-16,1.968750,1.979167,1.708333,1.729167,1.729167,14700000,1997-05
1997-05-19,1.760417,1.770833,1.625000,1.708333,1.708333,6106800,1997-05
1997-05-20,1.729167,1.750000,1.635417,1.635417,1.635417,5467200,1997-05
1997-05-21,1.635417,1.645833,1.375000,1.427083,1.427083,18853200,1997-05


In [6]:
# 현재 행의 STD-YM과 다음 행의 STD-YM이 다른 경우 → 월말
flag = df['STD-YM'] != df.shift(-1)['STD-YM']
df.loc[flag,]

,Open,High,Low,Close,Adj Close,Volume,STD-YM
Date,,,,,,,
1997-05-30,1.500000,1.510417,1.479167,1.500000,1.500000,2594400,1997-05
1997-06-30,1.510417,1.598958,1.479167,1.541667,1.541667,2746800,1997-06
1997-07-31,2.437500,2.437500,2.333333,2.395833,2.395833,1454400,1997-07
1997-08-29,2.364583,2.375000,2.322917,2.338542,2.338542,722400,1997-08
1997-09-30,4.000000,4.348958,3.802083,4.338542,4.338542,5254800,1997-09
...,...,...,...,...,...,...,...
2019-02-28,1635.250000,1651.770020,1633.829956,1639.829956,1639.829956,3025900,2019-02
2019-03-29,1786.579956,1792.859985,1776.630005,1780.750000,1780.750000,3320800,2019-03
2019-04-30,1930.099976,1935.709961,1906.949951,1926.520020,1926.520020,3506000,2019-04


In [7]:
month_last_df = df.groupby('STD-YM').tail(1)

In [8]:
# 전월의 수정 주가, 전년도의 수정 주가 컬럼 생성
month_last_df['BF-1M'] = month_last_df.shift(1)['Adj Close'].fillna(0)
month_last_df['BF-12M'] = month_last_df.shift(12)['Adj Close'].fillna(0)

In [9]:
month_last_df.head()

,Open,High,Low,Close,Adj Close,Volume,STD-YM,BF-1M,BF-12M
Date,,,,,,,,,
1997-05-30,1.500000,1.510417,1.479167,1.500000,1.500000,2594400,1997-05,0.000000,0.0
1997-06-30,1.510417,1.598958,1.479167,1.541667,1.541667,2746800,1997-06,1.500000,0.0
1997-07-31,2.437500,2.437500,2.333333,2.395833,2.395833,1454400,1997-07,1.541667,0.0
1997-08-29,2.364583,2.375000,2.322917,2.338542,2.338542,722400,1997-08,2.395833,0.0
1997-09-30,4.000000,4.348958,3.802083,4.338542,4.338542,5254800,1997-09,2.338542,0.0


5월 29일

In [ ]:
# 거래 내역을 df에 추가
# month_last = 구매 신호(momentum_index)를 확인하기 위함

for i in month_last_df.index:
    signal = ''

    # 절대 모멘텀의 계산식 → (전월의 수정 주가 / 전년의 수정 주가) - 1
    momentum_index = (month_last_df.loc[i, 'BF-1M'] / month_last_df.loc[i, 'BF-12M']) - 1

    # 0보다 크고 무한대가 아닌 경우가 구매 신호
    flag = (momentum_index > 0) & (momentum_index != np.inf)
    if flag:
        signal = 'buy'

    print(f'날짜: {i}, momentum_index: {momentum_index}, signal: {signal}')

    df.loc[i:, 'trade'] = signal

In [13]:
df['trade'].value_counts()

trade
buy    4033
       1520
Name: count, dtype: int64

In [15]:
df.tail()

,Open,High,Low,Close,Adj Close,Volume,STD-YM,trade
Date,,,,,,,,
2019-06-18,1901.349976,1921.670044,1899.790039,1901.369995,1901.369995,3895700,2019-06,buy
2019-06-19,1907.839966,1919.579956,1892.469971,1908.790039,1908.790039,2895300,2019-06,buy
2019-06-20,1933.329956,1935.199951,1905.800049,1918.189941,1918.189941,3217200,2019-06,buy
2019-06-21,1916.099976,1925.949951,1907.579956,1911.300049,1911.300049,3920300,2019-06,buy
2019-06-24,1912.660034,1916.859985,1901.329956,1907.953857,1907.953857,1243601,2019-06,buy


In [18]:
# 수익률, 누적 수익률 계산

df['rtn'] = 1.0

# 수익률 계산
for i in df.index:
    # 매수
    if (df.shift().loc[i, 'trade'] == '') & (df.loc[i, 'trade'] == 'buy'):
        buy = df.loc[i, 'Adj Close']
        print(f"매수일: {i}, 매수가: {buy}")
    elif (df.shift().loc[i, 'trade'] == 'buy') & (df.loc[i, 'trade'] == ''):
        sell = df.loc[i, 'Adj Close']
        rtn = sell / buy
        df.loc[i, 'rtn'] = rtn
        print(f"매도일: {i}, 매도가: {sell}, 수익률: {rtn}")
# 누적 수익률
df['acc_rtn'] = df['rtn'].cumprod()
# 최종 수익률
acc_rtn = df.iloc[-1, -1]

acc_rtn

매수일: 1998-05-29 00:00:00, 매수가: 7.34375
매도일: 2000-03-31 00:00:00, 매도가: 67.0, 수익률: 9.12340425531915
매수일: 2002-02-28 00:00:00, 매수가: 14.1
매도일: 2002-04-30 00:00:00, 매도가: 16.690001, 수익률: 1.183688014184397
매수일: 2002-06-28 00:00:00, 매수가: 16.25
매도일: 2004-08-31 00:00:00, 매도가: 38.139999, 수익률: 2.347076861538462
매수일: 2005-02-28 00:00:00, 매수가: 35.18
매도일: 2005-03-31 00:00:00, 매도가: 34.27, 수익률: 0.9741330301307563
매수일: 2005-08-31 00:00:00, 매수가: 42.700001
매도일: 2006-05-31 00:00:00, 매도가: 34.610001, 수익률: 0.8105386461232167
매수일: 2006-06-30 00:00:00, 매수가: 38.68
매도일: 2006-07-31 00:00:00, 매도가: 26.889999, 수익률: 0.6951912874870734
매수일: 2007-02-28 00:00:00, 매수가: 39.139999
매도일: 2008-07-31 00:00:00, 매도가: 76.339996, 수익률: 1.9504342859078763
매수일: 2009-06-30 00:00:00, 매수가: 83.660004
매도일: 2012-03-30 00:00:00, 매도가: 202.509995, 수익률: 2.420630950483818
매수일: 2012-04-30 00:00:00, 매수가: 231.899994
매도일: 2014-10-31 00:00:00, 매도가: 305.459991, 수익률: 1.31720568737919
매수일: 2015-03-31 00:00:00, 매수가: 372.100006


np.float64(86.52294619753461)

In [34]:
# df['Adj Close'].iloc[0]
df['Adj Close'][0]

KeyError: 0

In [33]:
# byandhold 수익률 계산

buy = df['Adj Close'].iloc[0]
sell = df['Adj Close'].iloc[-1]

# print(sell/rtn)
print(sell/buy)

974.2744757914


#### 절대 모멘텀 함수화

1. **STD-YM** 생성하는 함수
    - 매개변수
        - 데이터
        - 기준 column
    - df 깊은 복사
    - column에 Date 있는 지 확인, 있다면 Date를 인덱스로 변경
    - index를 시계열 데이터로 변경
    - tz 제거
    - 데이터에서 결측치와 무한대 제거
    - 기준 컬럼 제외 모두 제거
    - STD-YM column을 생성하여 index에서 년도-월 데이터를 추출하여 대입
    - 수정된 데이터 프레임을 되돌려준다.

In [ ]:
def create_ym(_df, _col = 'Adj Close'):
    df = _df.copy()
    # Date가 column에 포함되어 있는가?
    if 'Date' in df.columns:
        df.set_index('Date', inplace = True)
    df.index = pd.to_datetime(df.index)
    df.index = df.index.tz_localize(None)
    flag = df.isin([np.nan, np.inf, -np.inf]).any(axis=1)
    df = df.loc[~flag, [_col]]
    df['STD-YM'] = df.index.strftime('%Y-%m')

    return df

In [61]:
df = pd.read_csv('../csv/AAPL.csv')

In [62]:
ym_df = create_ym(df)
ym_df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 9713 entries, 1980-12-12 to 2019-06-24
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  9713 non-null   float64
 1   STD-YM     9713 non-null   str    
dtypes: float64(1), str(1)
memory usage: 227.6 KB


In [63]:
np.nan == np.nan

False

In [64]:
np.nan in [np.nan]

True

2. 월말 데이터를 생성하여 BF1, BF2 column을 생성하는 함수
    - 매개변수
        - `create_ym`의 결과를 받아주는 데이터
        - 시작 시간: 2010-01-01
        - 종료 시간: 현재 시간
        - momentum 기간: 12
        - 기준 시점: 1 (1인 경우 월말, 0인 경우 월초)
    - 기준 시점 값에 따라서 (월말|월초) 데이터만 모은 새로운 데이터프레임을 생성
    - 생성된 데이터프레임에서 BF1 column을 생성하여 전월의 데이터를 대입
    - BF2 column을 생성하여 momentum 기간(6 → 6개월 전) 전의 데이터를 대입
    - 결측치는 0으로 대체
    - DF을 시작시간과 종료시간으로 필터링
    - 결과를 return (월말 or 월초 DF)

In [65]:
def create_month(
        _df,
        _start = '2010-01-01',
        _end = datetime.now(),
        _momentum = 12,
        _last = 1
):

    if _last == 1:
        df = _df.groupby('STD-YM').tail(1)
    elif _last == 0:
        df = _df.groupby('STD-YM').head(1)
    else:
        return "_last 값은 0 또는 1만 가능합니다."
    
    col = _df.columns[0]

    df['BF1'] = df.shift(1)[col].fillna(0)
    df['BF2'] = df.shift(_momentum)[col].fillna(0)
    
    df = df.loc[ _start : _end, ]
    return df

In [70]:
month_df = create_month(ym_df)

In [71]:
month_df.head()

,Adj Close,STD-YM,BF1,BF2
Date,,,,
2010-01-29,24.035734,2010-01,26.372231,11.279500
2010-02-26,25.607582,2010-02,24.035734,11.176879
2010-03-31,29.409555,2010-03,25.607582,13.155455
2010-04-30,32.674633,2010-04,29.409555,15.747248
2010-05-28,32.147762,2010-05,32.674633,16.996212


3. 거래 내역을 추가하고 수익률을 계산하는 함수
    - 매개변수
        - ym_df를 대입할 수 있는 변수 (_df1)
        - month_df를 대입할 수 있는 변수 (_df2)
        - 모멘텀 스코어: 1
    - _df1 깊은 복사 (df)
    - df `trade` column 생성, 빈 텍스트 대입
    - df `rtn` column 생성, 1 대입
    - _df2를 이용하여 momentum index 생성
        - 0보다 크고 무한대가 아닌 경우 df의 보유 내역을 추가
    - 수익률 계산
    - 누적 수익률 계산
    - df와 최종 누적 수익률을 되돌려준다.

In [89]:
def create_rtn(_df1, _df2, _score = 1):
    df = _df1.copy()

    df['trade'] = ''
    df['rtn'] = 1.0

    col = df.columns[0]

    # _df2를 이용해서 거래 내역을 생성
    for i in _df2.index:
        signal = ''
        
        # momentum 계산
        momentum_index = _df2.loc[i, 'BF1'] / _df2.loc[i, 'BF2'] - _score
        flag = (momentum_index > 0) & (momentum_index != np.inf)
        
        if flag:
            signal = 'buy'
        
        # 거래 내역 생성
        df.loc[i:, 'trade'] = signal
        print(f'날짜: {i}, momentum_index: {momentum_index}, signal: {signal}')
    
    # 수익률 계산
    for i in df.index:
        if (df.shift(1).loc[i, 'trade'] == '') & (df.loc[i, 'trade'] == 'buy'):
            buy = df.loc[i, col]
            print(f'매수일: {i}, 매수가: {buy}')
        elif (df.shift(1).loc[i, 'trade'] == 'buy') & (df.loc[i, 'trade'] == ''):
            sell = df.loc[i, col]
            rtn = sell / buy
            df.loc[i, 'rtn'] = rtn
            print(f'매도일: {i}, 매도가: {sell}, 수익률: {rtn}')
    
    # 누적 수익률 계산
    df['acc_rtn'] = df['rtn'].cumprod()
    acc_rtn = df.iloc[-1, -1]

    return df, acc_rtn

In [90]:
df_final, acc_rtn = create_rtn(ym_df, month_df)

print(acc_rtn)

날짜: 2010-01-29 00:00:00, momentum_index: 1.338067378873177, signal: buy
날짜: 2010-02-26 00:00:00, momentum_index: 1.1504870903585878, signal: buy
날짜: 2010-03-31 00:00:00, momentum_index: 0.9465371589200071, signal: buy
날짜: 2010-04-30 00:00:00, momentum_index: 0.8675996593182504, signal: buy
날짜: 2010-05-28 00:00:00, momentum_index: 0.9224656058655893, signal: buy
날짜: 2010-06-30 00:00:00, momentum_index: 0.8035523759459491, signal: buy
날짜: 2010-07-30 00:00:00, momentum_index: 0.5394459942740937, signal: buy
날짜: 2010-08-31 00:00:00, momentum_index: 0.5293378740562196, signal: buy
날짜: 2010-09-30 00:00:00, momentum_index: 0.3115730863758013, signal: buy
날짜: 2010-10-29 00:00:00, momentum_index: 0.5053051244562894, signal: buy
날짜: 2010-11-30 00:00:00, momentum_index: 0.505577053958894, signal: buy
날짜: 2010-12-31 00:00:00, momentum_index: 0.476534124094393, signal: buy
날짜: 2011-01-31 00:00:00, momentum_index: 0.6794752346651864, signal: buy
날짜: 2011-02-28 00:00:00, momentum_index: 0.65829362569

In [91]:
ym_df

,Adj Close,STD-YM
Date,,
1980-12-12,0.410525,1980-12
1980-12-15,0.389106,1980-12
1980-12-16,0.360548,1980-12
1980-12-17,0.369472,1980-12
1980-12-18,0.380182,1980-12
...,...,...
2019-06-18,198.449997,2019-06
2019-06-19,197.869995,2019-06
2019-06-20,199.460007,2019-06
